- optimal control
    - We consider all of the work in optimal control also to be, in a sense, work in reinforcement learning. (sutton)
- $Q^*$ vs. $Q^\pi$
    - 智能体在诞生之初，手中只有一本极其糟糕的策略 $\pi_0$，以及对应的、充满误判的 $Q^{\pi_0}$（不带星号）。它必须通过不断与环境交互、试错，利用第二个公式（贝尔曼期望方程）不断更新自己的认知。整个训练循环的终极数学目标，就是通过无数次的迭代，将那个不带星号的、残缺的 $Q^{\pi_t}$，一步步推向、并最终在数学上等价于那个带星号的极限 $Q^*$。一旦 $Q^\pi = Q^*$，智能体就自动领悟了最优策略 $\pi^*$，训练宣告大功告成。

### definitions

$$
V^\pi(s) = \mathbb{E}_{\tau \sim \pi} [R(\tau) | s_0 = s]
$$
- 价值函数最本源的定义。它意味着：如果一个智能体站在当前状态 $s$ 往未来看，假设它接下来一直按照策略 $\pi$ 行动直到游戏结束，它平均能拿到的总分数，就是当前状态 $s$ 的价值。这是一种**“最终清算”**的思想。

$$
V^\pi(s) = \mathbb{E}_{\substack{a \sim \pi \\ s' \sim P}} [r(s, a) + \gamma V^\pi(s')]
$$
- 基于贝尔曼方程（Bellman Equation）的递推定义。当前状态的价值，等于**“眼前的收益”加上“打了折的下一个状态的价值”的期望。这是一种“走一步看一步”**的思想。
- why bellman equation
    - 不可计算性（Intractability）： 第一条公式在现实中几乎是无法直接计算的。对于稍微复杂的任务（比如下围棋、自动驾驶），未来的轨迹 $\tau$ 有无数种分支可能，直到尽头才能结算 $R(\tau)$，计算成本属于天文数字。
    - 动态规划的基石（Dynamic Programming）： 第二条公式（贝尔曼方程）使得强化学习脱离了穷举法。通过将 $V^\pi(s)$ 和 $V^\pi(s')$ 建立递归关系，算法可以利用**“自举（Bootstrapping）”**的思想，用对后续状态的估计来更新对当前状态的估计。诸如 Q-Learning、时序差分（TD）算法、策略迭代（Policy Iteration）等现代 RL 算法，其核心原理全都是在不断让公式等号左右两边逼近相等。
        - bootstrapping：Updating a guess based on another guess
        - 智能体在状态 $s$ 走了一步，进入状态 $s'$，拿到即刻的奖励 $r$。此时游戏远未结束，但它不打算等了(rollout 到结束)。它直接调出大脑中对 $s'$ 的已有评估（即 $V^\pi(s')$，这本质上依然是个猜测），将 $r + \gamma V^\pi(s')$ 组合成一个“伪真实目标”，用来更新对当前状态 $s$ 的评估。
    - 马尔可夫性（Markov Property）： 第二个公式能成立的前提，是环境必须满足马尔可夫性——即“未来只与现在有关，与过去无关”。$s'$ 完美浓缩了之前发生的一切信息。

### bellman equation

$$
\begin{split}
Q^{*}(s,a)&=\sum_{s'}P(s'|s,a)[R(s,a,s')+\gamma\max_{a'}Q^*(s',a')]\\
&=\mathbb E_{s'\sim P(\cdot|s,a)}[R(s,a,s')+\gamma\max_{a'}Q^*(s',a')]
\end{split}
$$

$$
V^*(s) = \max_a \sum_{s'} P(s'|s, a) \left( R(s, a, s') + \gamma V^*(s') \right)
$$

- $Q^*(s,a)$: optimal action-value function
- $\max_{a'}Q^*(s',a')$: 下一个状态下的最大价值
- $R(s,a,s')$: 当前能获得的即时奖励，$\gamma\max_{a'}Q^*(s',a')$: 未来能获得的最大奖励（折现）
- 重新理解这个等式
    - 等式左边：在状态 $s$，选择了动作 $a$，则总回报是多少？
    - 等式右边：即时奖励 + 遍历所有 $s'$ 对应的最大价值（折现）
- 期望是因为转移是不确定的，当确定性转移时
    - $V^*(s) = \max_a \left( R(s, a) + \gamma V^*(s') \right)$

### q-learning

> model free：不需要知道 $P(s'|s,a)$ 和奖励 $R(s,a,s')$，通过与环境的交互、采样数据来学习；

$$
\begin{split}
Q(s,a)&\leftarrow Q(s,a)+\alpha(r+\gamma \max_{a'}Q(s',a')-Q(s,a))\\
Q(s,a)&\leftarrow (1-\alpha)Q(s,a)+\alpha(r+\gamma \max_{a'}Q(s',a'))
\end{split}
$$

- $r+\gamma \max_{a'}Q(s',a')$: TD target
    - $r$ 是对 $R(s,a,s')$ 的一次采样（跟环境交互得到的）
    - $\max_{a'}Q(s',a')$ 则是对 $\max_{a'}Q^*(s',a')$ 的当前估计
- $\alpha$: 表示 update 时的 learning rate，
    - $(1-\alpha)Q(s,a)$: 旧知识的保留
    - $\alpha(r+\gamma \max_{a'}Q(s',a'))$: 新知识的加入